In [25]:
import os
import sys
from pathlib import Path

import hydra
import rootutils
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from torchvision.transforms import v2
from omegaconf import OmegaConf
from typing import Dict, List, Any

# Setup root directory
project_root = rootutils.setup_root(
    __file__ if "__file__" in dir() else os.getcwd(),
    indicator=".project-root",
    pythonpath=True,
)

src_root = project_root / "src"

if src_root and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
 
print(sys.path)  

from utils.utils import (
    calculate_summary_statistics,
    collect_preds_targets,
    create_confusion_matrix,
    get_class_names,
    to_float,
) 

['/home/lukasb/Documents/NoisyLabelDefectDetection', '/home/lukasb/Documents/NoisyLabelDefectDetection', '/home/lukasb/Documents/NoisyLabelDefectDetection', '/home/lukasb/Documents/NoisyLabelDefectDetection/src', '/home/lukasb/Documents/NoisyLabelDefectDetection', '/home/lukasb/miniforge3/envs/defect-detection/lib/python310.zip', '/home/lukasb/miniforge3/envs/defect-detection/lib/python3.10', '/home/lukasb/miniforge3/envs/defect-detection/lib/python3.10/lib-dynload', '', '/home/lukasb/.local/lib/python3.10/site-packages', '/home/lukasb/miniforge3/envs/defect-detection/lib/python3.10/site-packages', '/tmp/tmp_5hn_dph']


## Configuration

Point to the log directory you want to evaluate.

In [26]:
# Paths
LOG_ROOT = Path("/home/lukasb/Documents/NoisyLabelDefectDetection/logs/train/SurfaceDefectDetection/noisy_new/2026-02-24_05-32-10_ce").resolve()
CONFIG_PATH = "../configs"
CONFIG_NAME = "train.yaml"

def select_best_checkpoint(run_dir: Path) -> Path:
    ckpt_dir = run_dir / "checkpoints"
    ckpts = list(ckpt_dir.glob("epoch_*-val_f1_*.ckpt"))
    if ckpts:
        def parse_f1(path: Path) -> float:
            try:
                return float(path.stem.split("val_f1_")[-1])
            except ValueError:
                return -1.0
        return max(ckpts, key=parse_f1)
    last_ckpt = ckpt_dir / "last.ckpt"
    if last_ckpt.exists():
        return last_ckpt
    raise FileNotFoundError(f"No checkpoints found in {ckpt_dir}")

## Load Configuration and Data

In [28]:
# Load configuration
from hydra.core.global_hydra import GlobalHydra
from lightning import Callback, Trainer

if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

with hydra.initialize(version_base="1.3", config_path=CONFIG_PATH):
    cfg = hydra.compose(config_name=CONFIG_NAME)

hyperparams_path = LOG_ROOT / "summary" / "hyperparameters.yaml"
if hyperparams_path.exists():
    hyperparams = OmegaConf.load(hyperparams_path)
    print("Loaded hyperparameters from experiment")
else:
    hyperparams = cfg
    print("Using default config (hyperparameters.yaml not found)")
    
cfg = hyperparams

Loaded hyperparameters from experiment


In [ ]:
def run_validation(cfg, run_idx, seed, checkpoint_path) -> Dict[str, Any]:
    # Data module
    datamodule = hydra.utils.instantiate(cfg.data, seed=seed)
    datamodule.prepare_data()
    datamodule.setup(stage="fit")
    class_names = datamodule.class_names

    model = hydra.utils.instantiate(cfg.model, datamodule=datamodule)

    val_trainer: Trainer = hydra.utils.instantiate(
        cfg.trainer,
        callbacks=[],
        devices=1,
        strategy="auto",
        
    )

    val_trainer.validate(
        model=model,
        datamodule=datamodule,
        ckpt_path=checkpoint_path,
        weights_only=False,
    )

    class_names = get_class_names(datamodule)

    cm = val_trainer.callback_metrics

    val_metrics: Dict[str, Any] = {
        "run_idx": run_idx,
        # Primary reporting (imbalance + equal class importance)
        "val/f1_macro": to_float(cm.get("val/f1_macro")),
        "val/precision_macro": to_float(cm.get("val/precision_macro")),
        "val/recall_macro": to_float(cm.get("val/recall_macro")),
        # Context / secondary metrics
        "val/acc": to_float(cm.get("val/acc")),
        "val/f1_weighted": to_float(cm.get("val/f1_weighted")),
        "val/precision_weighted": to_float(cm.get("val/precision_weighted")),
        "val/recall_weighted": to_float(cm.get("val/recall_weighted")),
        "val/loss": to_float(cm.get("val/loss")),
    }

    n_classes = None
    try:
        n_classes = int(getattr(datamodule, "num_classes"))
    except Exception:
        n_classes = None

    if n_classes is None and class_names:
        n_classes = len(class_names)

    class_names_for_metrics = None
    if class_names and n_classes is not None and len(class_names) >= n_classes:
        class_names_for_metrics = class_names

    if n_classes is not None and n_classes > 0:
        for i in range(n_classes):
            for metric_name in ("precision", "recall", "f1"):
                key_idx = f"val/{metric_name}_c{i}"
                if class_names_for_metrics:
                    class_name = class_names_for_metrics[i]
                    key_named = f"val/{metric_name}_{class_name}"
                    if key_named in cm:
                        val_metrics[key_named] = to_float(cm.get(key_named))
                    elif key_idx in cm:
                        val_metrics[key_named] = to_float(cm.get(key_idx))
                else:
                    if key_idx in cm:
                        val_metrics[key_idx] = to_float(cm.get(key_idx))

    return val_metrics

In [32]:
runs = 10

all_metrics: List[Dict[str, Any]] = []

for run_idx in range(1, runs + 1):
    seed = 41 + run_idx
    run_dir = LOG_ROOT / f"run_{run_idx}_seed_{seed}"
    checkpoint_path = select_best_checkpoint(run_dir)

    print(f"Checkpoint: {checkpoint_path}")
    print(f"Run idx: {run_idx}")
    print(f"Run seed: {seed}")

    val_metrics = run_validation(cfg, run_idx, seed, checkpoint_path)
    all_metrics.append(val_metrics)
    

all_metrics_df, summary_df = calculate_summary_statistics(all_metrics)

summary_dir = LOG_ROOT / "summary"
summary_dir.mkdir(parents=True, exist_ok=True)
all_metrics_df.to_csv(summary_dir / "all_runs_metrics.csv", index=False)
summary_df.to_csv(summary_dir / "summary_statistics.csv", index=False)

print(f"\nSummary Statistics:\n{summary_df.to_string()}")

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
Restoring states from the checkpoint path at /home/lukasb/Documents/NoisyLabelDefectDetection/logs/train/SurfaceDefectDetection/noisy_new/2026-02-24_05-32-10_ce/run_1_seed_42/checkpoints/epoch_130-val_f1_0.6561.ckpt


/home/lukasb/miniforge3/envs/defect-detection/lib/python3.10/site-packages/lightning/pytorch/trainer/call.py:283: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["ModelCheckpoint{'monitor': 'val/f1_macro', 'mode': 'max', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}", "EarlyStopping{'monitor': 'val/f1_macro', 'mode': 'max'}"].
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /home/lukasb/Documents/NoisyLabelDefectDetection/logs/train/SurfaceDefectDetection/noisy_new/2026-02-24_05-32-10_ce/run_1_seed_42/checkpoints/epoch_130-val_f1_0.6561.ckpt


Checkpoint: /home/lukasb/Documents/NoisyLabelDefectDetection/logs/train/SurfaceDefectDetection/noisy_new/2026-02-24_05-32-10_ce/run_1_seed_42/checkpoints/epoch_130-val_f1_0.6561.ckpt
Run idx: 1
Run seed: 42
Classes: ['background', 'black_stain', 'corrosion', 'crack', 'deformation', 'missing_part', 'ok', 'other', 'silicate_stain', 'water_stain']


/home/lukasb/miniforge3/envs/defect-detection/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃       Validate metric        ┃         DataLoader 0         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│           val/acc            │      0.6752806901931763      │
│      val/f1_background       │      0.9174311757087708      │
│      val/f1_black_stain      │      0.6823529601097107      │
│       val/f1_corrosion       │      0.6878306865692139      │
│         val/f1_crack         │      0.5287356376647949      │
│      val/f1_deformation      │      0.6145251393318176      │
│         val/f1_macro         │      0.6561429500579834      │
│      val/f1_macro_best       │      0.6561429500579834      │
│     val/f1_missing_part      │      0.752136766910553       │
│          val/f1_ok           │      0.5040000081062317      │
│         val/f1_other         │      0.7368420958518982      │
│    val/f1_silicate_stain     │     0.40816327929496765      │
│      val/f1_water_stain      │      0.729411780834198       │
│       val/f1_weighted        │      0.624488115310669       │
│           val/loss           │      2.1600687503814697      │
│   val/precision_background   │      0.9090909361839294      │
│  val/precision_black_stain   │      0.5471698045730591      │
│   val/precision_corrosion    │      0.6190476417541504      │
│     val/precision_crack      │      0.4893617033958435      │
│  val/precision_deformation   │           0.859375           │
│     val/precision_macro      │      0.7045751810073853      │
│  val/precision_missing_part  │      0.7719298005104065      │
│       val/precision_ok       │      0.3962264060974121      │
│     val/precision_other      │      0.8235294222831726      │
│ val/precision_silicate_stain │      0.9090909361839294      │
│  val/precision_water_stain   │      0.7209302186965942      │
│    val/precision_weighted    │      0.7073572874069214      │
│    val/recall_background     │      0.9259259104728699      │
│    val/recall_black_stain    │           0.90625            │
│     val/recall_corrosion     │      0.773809552192688       │
│       val/recall_crack       │      0.574999988079071       │
│    val/recall_deformation    │       0.47826087474823       │
│       val/recall_macro       │      0.6752806901931763      │
│   val/recall_missing_part    │      0.7333333492279053      │
│        val/recall_ok         │      0.692307710647583       │
│       val/recall_other       │      0.6666666865348816      │
│  val/recall_silicate_stain   │      0.2631579041481018      │
│    val/recall_water_stain    │      0.738095223903656       │
│     val/recall_weighted      │      0.6202672719955444      │
└──────────────────────────────┴──────────────────────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃       Validate metric        ┃         DataLoader 0         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│           val/acc            │      0.6849230527877808      │
│      val/f1_background       │      0.9230769276618958      │
│      val/f1_black_stain      │      0.6428571343421936      │
│       val/f1_corrosion       │      0.7804877758026123      │
│         val/f1_crack         │      0.5274725556373596      │
│      val/f1_deformation      │      0.650994598865509       │
│         val/f1_macro         │      0.6741695404052734      │
│      val/f1_macro_best       │      0.6741695404052734      │
│     val/f1_missing_part      │      0.7457627058029175      │
│          val/f1_ok           │      0.5425742864608765      │
│         val/f1_other         │      0.699999988079071       │
│    val/f1_silicate_stain     │     0.48148149251937866      │
│      val/f1_water_stain      │      0.7469879388809204      │
│       val/f1_weighted        │      0.6564973592758179      │
│           val/loss           │      1.9546164274215698      │
│   val/precision_background   │      0.9599999785423279      │
│  val/precision_black_stain   │      0.5192307829856873      │
│   val/precision_corrosion    │      0.800000011920929       │
│     val/precision_crack      │     0.47058823704719543      │
│  val/precision_deformation   │      0.8653846383094788      │
│     val/precision_macro      │      0.7103412747383118      │
│  val/precision_missing_part  │      0.7586206793785095      │
│       val/precision_ok       │      0.4241486191749573      │
│     val/precision_other      │      0.7368420958518982      │
│ val/precision_silicate_stain │            0.8125            │
│  val/precision_water_stain   │      0.7560975551605225      │
│    val/precision_weighted    │      0.7281221747398376      │
│    val/recall_background     │      0.8888888955116272      │
│    val/recall_black_stain    │           0.84375            │
│     val/recall_corrosion     │      0.761904776096344       │
│       val/recall_crack       │      0.6000000238418579      │
│    val/recall_deformation    │       0.52173912525177       │
│       val/recall_macro       │      0.6849230527877808      │
│   val/recall_missing_part    │      0.7333333492279053      │
│        val/recall_ok         │      0.7527472376823425      │
│       val/recall_other       │      0.6666666865348816      │
│  val/recall_silicate_stain   │     0.34210526943206787      │
│    val/recall_water_stain    │      0.738095223903656       │
│     val/recall_weighted      │      0.6481069326400757      │
└──────────────────────────────┴──────────────────────────────┘

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
Restoring states from the checkpoint path at /home/lukasb/Documents/NoisyLabelDefectDetection/logs/train/SurfaceDefectDetection/noisy_new/2026-02-24_05-32-10_ce/run_10_seed_51/checkpoints/epoch_066-val_f1_0.6506.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /home/lukasb/Documents/NoisyLabelDefectDetection/logs/train/SurfaceDefectDetection/noisy_new/2026-02-24_05-32-10_ce/run_10_seed_51/checkpoints/epoch_066-val_f1_0.6506.ckpt


Checkpoint: /home/lukasb/Documents/NoisyLabelDefectDetection/logs/train/SurfaceDefectDetection/noisy_new/2026-02-24_05-32-10_ce/run_10_seed_51/checkpoints/epoch_066-val_f1_0.6506.ckpt
Run idx: 10
Run seed: 51
Classes: ['background', 'black_stain', 'corrosion', 'crack', 'deformation', 'missing_part', 'ok', 'other', 'silicate_stain', 'water_stain']


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃       Validate metric        ┃         DataLoader 0         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│           val/acc            │      0.6673579216003418      │
│      val/f1_background       │      0.9259259104728699      │
│      val/f1_black_stain      │      0.6753246784210205      │
│       val/f1_corrosion       │      0.6847826242446899      │
│         val/f1_crack         │      0.5249999761581421      │
│      val/f1_deformation      │      0.6268116235733032      │
│         val/f1_macro         │      0.6506335735321045      │
│      val/f1_macro_best       │      0.6506335735321045      │
│     val/f1_missing_part      │      0.6433566212654114      │
│          val/f1_ok           │      0.5291666388511658      │
│         val/f1_other         │      0.699999988079071       │
│    val/f1_silicate_stain     │     0.44897958636283875      │
│      val/f1_water_stain      │      0.7469879388809204      │
│       val/f1_weighted        │      0.6285374164581299      │
│           val/loss           │      1.8346412181854248      │
│   val/precision_background   │      0.9259259104728699      │
│  val/precision_black_stain   │      0.5777778029441833      │
│   val/precision_corrosion    │      0.6299999952316284      │
│     val/precision_crack      │      0.5249999761581421      │
│  val/precision_deformation   │      0.8357487916946411      │
│     val/precision_macro      │      0.6967783570289612      │
│  val/precision_missing_part  │      0.5542168617248535      │
│       val/precision_ok       │     0.42617449164390564      │
│     val/precision_other      │      0.7368420958518982      │
│ val/precision_silicate_stain │             1.0              │
│  val/precision_water_stain   │      0.7560975551605225      │
│    val/precision_weighted    │      0.6979830265045166      │
│    val/recall_background     │      0.9259259104728699      │
│    val/recall_black_stain    │            0.8125            │
│     val/recall_corrosion     │             0.75             │
│       val/recall_crack       │      0.5249999761581421      │
│    val/recall_deformation    │      0.5014492869377136      │
│       val/recall_macro       │      0.6673579216003418      │
│   val/recall_missing_part    │      0.7666666507720947      │
│        val/recall_ok         │      0.6978021860122681      │
│       val/recall_other       │      0.6666666865348816      │
│  val/recall_silicate_stain   │     0.28947368264198303      │
│    val/recall_water_stain    │      0.738095223903656       │
│     val/recall_weighted      │      0.6258351802825928      │
└──────────────────────────────┴──────────────────────────────┘


Summary Statistics:
                          metric      mean    median       std  ci_lower  ci_upper       min       max
0                   val/f1_macro  0.652247  0.650524  0.012133  0.643568  0.660927  0.635021  0.674170
1            val/precision_macro  0.688561  0.693952  0.016318  0.676888  0.700234  0.662213  0.710341
2               val/recall_macro  0.668605  0.669869  0.013473  0.658967  0.678243  0.637548  0.684923
3                        val/acc  0.668605  0.669869  0.013473  0.658967  0.678243  0.637548  0.684923
4                val/f1_weighted  0.625369  0.623442  0.018332  0.612255  0.638483  0.595049  0.656497
5         val/precision_weighted  0.694713  0.699804  0.026829  0.675521  0.713905  0.628953  0.728122
6            val/recall_weighted  0.620713  0.618040  0.017865  0.607933  0.633493  0.592428  0.648107
7                       val/loss  2.073819  2.034061  0.195308  1.934104  2.213535  1.834641  2.409382
8       val/precision_background  0.923603  0.925926